# HI 5312 SQL Week 1: From CSV files to PostgreSQL tables

**Purpose:** This notebook is the lab guide for our first SQL session. Most SQL will be run in the **Codespaces Terminal** using `psql`. Use this notebook to follow the sequence, copy commands, record answers, and complete short practice checks.

**Datasets used today**

- `patients.csv`
- `bp_readings.csv`
- `medications.csv`

**Central question for every step:** *What does one row represent?*

## 0. Session flow

| Time | Activity | Student action |
|---|---|---|
| 0–8 min | Inspect CSV files | Identify rows, columns, keys, and data types |
| 8–18 min | Connect to PostgreSQL | Use terminal and `psql` |
| 18–32 min | Create tables | Define variables and SQL data types |
| 32–42 min | Load CSV files | Use `\copy` and validate row counts |
| 42–56 min | Simple queries | `SELECT`, `WHERE`, `ORDER BY`, `COUNT`, `GROUP BY` |
| 56–60 min | First `JOIN` | Connect patients to BP readings and medications |

## 1. Inspect the source files first

Before SQL, inspect the CSV files. In health informatics, this is not optional. You need to know what the fields mean before you write queries.

**In the Codespaces Terminal, run:**

```bash
head -5 data/patients.csv
head -5 data/bp_readings.csv
head -5 data/medications.csv
```

**Discussion prompts**

1. What does one row represent in `patients.csv`?
2. What does one row represent in `bp_readings.csv`?
3. What does one row represent in `medications.csv`?
4. Which field connects all three files?

In [ ]:
# Optional: Use Python in this notebook to preview the CSV files.
from pathlib import Path
import pandas as pd

DATA_DIR = Path('/workspaces/hi5304-notebooks/data')

for file_name in ['patients.csv', 'bp_readings.csv', 'medications.csv']:
    path = DATA_DIR / file_name
    print('
' + '=' * 70)
    print(file_name, 'exists:', path.exists())
    if path.exists():
        df = pd.read_csv(path)
        print('Rows, columns:', df.shape)
        print('Columns:', list(df.columns))
        display(df.head())

## 2. Connect to PostgreSQL with `psql`

Most of today’s SQL should be run from the **Codespaces Terminal**, not inside this notebook.

**In the terminal:**

```bash
psql -h db -U hi5304_user -d hi5304
```

If prompted, enter the database password.

When you see this prompt, you are inside PostgreSQL:

```text
hi5304=#
```

Try these commands:

```sql
SELECT current_database();
\dt
```

To exit later:

```sql
\q
```

## 3. Optional local database concept

In Codespaces, the course database already exists. On a local PC using pgAdmin or SQL Shell, the database creation step would look like this:

```sql
CREATE DATABASE hi5312_demo;
```

Then in `psql`:

```sql
\c hi5312_demo
```

For Codespaces, stay in the existing `hi5304` database and create the tables there.

## 4. SQL data types: defining variables

When you create a PostgreSQL table, you must define the columns and their data types.

| CSV field | Example | SQL type | Why |
|---|---:|---|---|
| `patient_id` | `1001` | `INTEGER` | Whole-number identifier |
| `first_name` | `John` | `TEXT` | Text value |
| `date_of_birth` | `1968-04-12` | `DATE` | Calendar date |
| `systolic` | `148` | `INTEGER` | Whole-number BP value |
| `dose` | `10 mg` | `TEXT` | Includes both number and unit |
| `zip_code` | `75201` | `TEXT` | Identifier, not a measurement |

**Practice:** Why should ZIP code be `TEXT` rather than `INTEGER`?

## 5. Create the `patients` table

Copy and paste this block into the **psql terminal**:

```sql
DROP TABLE IF EXISTS patients;

CREATE TABLE patients (
    patient_id     INTEGER PRIMARY KEY,
    first_name     TEXT,
    last_name      TEXT,
    date_of_birth  DATE,
    sex            TEXT,
    race           TEXT,
    ethnicity      TEXT,
    zip_code       TEXT
);
```

Then check the table structure:

```sql
\d patients
```

**Question:** What does `PRIMARY KEY` mean in this table?

## 6. Load `patients.csv` into PostgreSQL

In the **psql terminal**, run:

```sql
\copy patients
FROM '/workspaces/hi5304-notebooks/data/patients.csv'
WITH (FORMAT csv, HEADER true);
```

Validate the import:

```sql
SELECT COUNT(*) FROM patients;
SELECT * FROM patients LIMIT 5;
```

**Expected row count:** 5.

**Important:** `HEADER true` skips the header row, but it does **not** automatically map columns by name. The CSV column order must match the table column order.

## 7. Create and load `bp_readings`

Run this in the **psql terminal**:

```sql
DROP TABLE IF EXISTS bp_readings;

CREATE TABLE bp_readings (
    bp_id         INTEGER PRIMARY KEY,
    patient_id    INTEGER,
    reading_date  DATE,
    systolic      INTEGER,
    diastolic     INTEGER,
    heart_rate    INTEGER,
    source        TEXT
);
```

Then load the CSV:

```sql
\copy bp_readings
FROM '/workspaces/hi5304-notebooks/data/bp_readings.csv'
WITH (FORMAT csv, HEADER true);
```

Validate:

```sql
SELECT COUNT(*) FROM bp_readings;
SELECT * FROM bp_readings LIMIT 5;
```

**Question:** Why can the same `patient_id` appear more than once in `bp_readings`?

## 8. Create and load `medications`

Run this in the **psql terminal**:

```sql
DROP TABLE IF EXISTS medications;

CREATE TABLE medications (
    med_id            INTEGER PRIMARY KEY,
    patient_id        INTEGER,
    medication_name   TEXT,
    dose              TEXT,
    frequency         TEXT,
    start_date        DATE,
    indication        TEXT
);
```

Then load the CSV:

```sql
\copy medications
FROM '/workspaces/hi5304-notebooks/data/medications.csv'
WITH (FORMAT csv, HEADER true);
```

Validate:

```sql
SELECT COUNT(*) FROM medications;
SELECT * FROM medications LIMIT 5;
```

**Practice:** What would happen if `dose` were defined as `INTEGER`?

## 9. Basic query pattern: `SELECT`, `FROM`, `LIMIT`

Run these in **psql**:

```sql
SELECT *
FROM patients
LIMIT 5;
```

```sql
SELECT patient_id, first_name, last_name, zip_code
FROM patients
LIMIT 5;
```

**Practice:** Write a query that returns only `patient_id`, `sex`, `race`, and `ethnicity` from `patients`.

### Your answer

Paste your query here:

```sql

```

## 10. Filter rows with `WHERE`

Run these in **psql**:

```sql
SELECT *
FROM bp_readings
WHERE systolic >= 140;
```

```sql
SELECT *
FROM medications
WHERE indication = 'Hypertension';
```

**Practice:** Find all home blood pressure readings.

Hint: use the `source` column.

### Your answer

```sql

```

## 11. Combine conditions with `AND`

Find clinic blood pressure readings where systolic BP is at least 140:

```sql
SELECT *
FROM bp_readings
WHERE source = 'Clinic'
  AND systolic >= 140;
```

**Practice:** Find home BP readings where diastolic BP is at least 85.

### Your answer

```sql

```

## 12. Sort results with `ORDER BY`

```sql
SELECT patient_id, reading_date, systolic, diastolic
FROM bp_readings
ORDER BY patient_id, reading_date;
```

Try descending systolic BP:

```sql
SELECT patient_id, reading_date, systolic, diastolic
FROM bp_readings
ORDER BY systolic DESC;
```

**Question:** Why is sorting useful before trusting a dataset?

## 13. Explore categories with `DISTINCT`

```sql
SELECT DISTINCT source
FROM bp_readings;
```

```sql
SELECT DISTINCT indication
FROM medications;
```

**Practice:** What distinct values appear in `sex` in the `patients` table?

### Your answer

```sql

```

## 14. Count and summarize with `GROUP BY`

How many BP readings does each patient have?

```sql
SELECT patient_id, COUNT(*) AS n_readings
FROM bp_readings
GROUP BY patient_id
ORDER BY n_readings DESC;
```

How many medications by indication?

```sql
SELECT indication, COUNT(*) AS n_meds
FROM medications
GROUP BY indication
ORDER BY n_meds DESC;
```

**Key point:** `GROUP BY` changes the grain. Before grouping, one row may be one medication. After grouping, one row may be one indication.

## 15. First join: patients + blood pressure readings

```sql
SELECT
    p.patient_id,
    p.first_name,
    p.last_name,
    b.reading_date,
    b.systolic,
    b.diastolic
FROM patients AS p
JOIN bp_readings AS b
    ON p.patient_id = b.patient_id
ORDER BY p.patient_id, b.reading_date;
```

**Questions**

1. Before the join, how many rows are in `patients`?
2. Before the join, how many rows are in `bp_readings`?
3. After the join, what does one row represent?

## 16. Three-table join: patients + BP + medications

This query asks a richer question, but it also creates a row-multiplication risk because a patient may have multiple BP readings and multiple medications.

```sql
SELECT
    p.patient_id,
    p.first_name,
    p.last_name,
    b.systolic,
    b.diastolic,
    m.medication_name,
    m.indication
FROM patients AS p
LEFT JOIN bp_readings AS b
    ON p.patient_id = b.patient_id
LEFT JOIN medications AS m
    ON p.patient_id = m.patient_id
WHERE b.systolic >= 140
ORDER BY p.patient_id;
```

**Discussion:** What does one row represent now?

## 17. Debugging checklist

When an import or query fails, ask:

1. Am I inside `psql` or the regular terminal?
2. Did I create the table first?
3. Do the CSV columns match the table columns in number and order?
4. Do the SQL data types match the actual values?
5. Did I include `HEADER true`?
6. Did I use the correct file path?
7. Did I validate with `COUNT(*)` and `LIMIT`?

**Useful commands**

```sql
\dt
\d patients
SELECT COUNT(*) FROM patients;
SELECT * FROM patients LIMIT 5;
```

## 18. Exit ticket

Before leaving class, write one query and explain it in plain English.

**My SQL query**

```sql

```

**Plain-language explanation**

> 

**What does one row represent in my result?**

>